# routes/verify

> Route handler for verify selection + audio extraction from video sources

In [ ]:
#| default_exp routes.verify

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import logging
from typing import Dict, Tuple, Callable

from fasthtml.common import APIRouter

from cjm_workflow_state.state_store import SQLiteWorkflowStateStore

from cjm_transcription_source_select.models import SourceSelectUrls, ExtractionResult
from cjm_transcription_source_select.routes.core import (
    get_step_state, update_step_state, get_session_id_from_sess
)
from cjm_transcription_source_select.components.stats_panel import render_stats_panel
from cjm_transcription_source_select.components.selection_panel import render_selection_panel
from cjm_transcription_source_select.services.source_select import SourceSelectService

logger = logging.getLogger(__name__)

## Verify Handler

Verifies the selection by:
1. Identifying video files that need audio extraction
2. Extracting audio from each video via the FFmpeg plugin (stream copy — nearly instant)
3. Building `committed_audio_paths` (audio files keep their original path, video files use extracted audio path)
4. Setting `verified = True`

Already-extracted videos (from a previous verify) are not re-extracted.

In [ ]:
#| export
async def _handle_verify(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    workflow_id: str,  # Workflow identifier
    urls: SourceSelectUrls,  # URL bundle
    service: SourceSelectService,  # Source select service
    sess,  # FastHTML session
):  # (stats panel, OOB selection panel)
    """Verify selection and extract audio from video files."""
    session_id = get_session_id_from_sess(sess)
    step_state = get_step_state(state_store, workflow_id, session_id)
    selected_files = step_state.get("selected_files", [])
    extraction_results = step_state.get("extraction_results", {})

    if not selected_files:
        return render_stats_panel(selected_files, urls)

    # Extract audio from video files
    for f in selected_files:
        if f.get("file_type") != "video":
            continue
        path = f["path"]

        # Skip already-extracted
        existing = extraction_results.get(path)
        if existing and existing.get("status") == "complete":
            continue

        # Extract audio
        try:
            result = await service.extract_audio(path)
            extraction_results[path] = ExtractionResult(
                status="complete",
                audio_path=result["output_path"],
                job_id=result.get("job_id"),
            )
            logger.info(f"Extracted audio from {path} -> {result['output_path']}")
        except Exception as e:
            extraction_results[path] = ExtractionResult(
                status="error",
                error=str(e),
            )
            logger.error(f"Failed to extract audio from {path}: {e}")

    # Build committed_audio_paths
    committed_audio_paths = []
    all_ok = True
    for f in selected_files:
        path = f["path"]
        if f.get("file_type") == "audio":
            committed_audio_paths.append(path)
        elif f.get("file_type") == "video":
            result = extraction_results.get(path, {})
            if result.get("status") == "complete":
                committed_audio_paths.append(result["audio_path"])
            else:
                all_ok = False

    # Only mark verified if all extractions succeeded
    verified = all_ok

    update_step_state(
        state_store, workflow_id, session_id,
        extraction_results=extraction_results,
        committed_audio_paths=committed_audio_paths if verified else [],
        verified=verified,
    )

    # Return updated stats panel + OOB selection panel with extraction statuses
    stats = render_stats_panel(selected_files, urls, extraction_results, verified)
    selection = render_selection_panel(selected_files, urls, extraction_results)
    selection.attrs["hx-swap-oob"] = "outerHTML"
    return stats, selection

## Router Initialization

In [ ]:
#| export
def init_verify_router(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    workflow_id: str,  # Workflow identifier
    urls: SourceSelectUrls,  # Mutable URL bundle
    service: SourceSelectService,  # Source select service
    prefix: str = "/verify",  # Route prefix
) -> Tuple[APIRouter, Dict[str, Callable]]:  # (router, route_dict)
    """Initialize verify route for selection verification and audio extraction."""
    router = APIRouter(prefix=prefix)

    @router
    async def verify(sess):
        """Verify selection and extract audio from video files."""
        return await _handle_verify(state_store, workflow_id, urls, service, sess)

    routes = {
        "verify": verify,
    }

    return router, routes

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()